# Explanation Generation
I need to use the model to get *k* explanations for each line of generated code. Firtly, I need to delete the comments line from my dataset.

In [1]:
import re
import os
import pandas as pd
from tqdm import tqdm
from enum import Enum
from transformers import AutoTokenizer, AutoModelForCausalLM

In [30]:
python_csv_file = '/datasets/leetcode_selected_problems_python_14B_no_comments.csv'
c_sharp_csv_file = '/datasets/leetcode_selected_problems_c_sharp_14B_no_comments.csv'
typescript_csv_file = '/datasets/leetcode_selected_problems_typescript_14B_no_comments.csv'

def get_generated_code(csv_file):
    csv_full_path = os.getcwd()+csv_file
    if os.path.exists(csv_full_path):
        return pd.read_csv(csv_full_path)
    else:
        print("File not found.")
        return None
    
python_solutions = get_generated_code(python_csv_file)
c_sharp_solutions = get_generated_code(c_sharp_csv_file)
typescript_solutions = get_generated_code(typescript_csv_file)

In [3]:
class Language(Enum):
    PYTHON = "python"
    TYPESCRIPT = "typescript"
    C_SHARP = "csharp"

In [4]:
model_name = "Qwen/Qwen2.5-Coder-14B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name, temperature=0.8)

2025-03-22 13:15:05.879727: I tensorflow/core/util/port.cc:111] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-03-22 13:15:05.925508: E tensorflow/compiler/xla/stream_executor/cuda/cuda_dnn.cc:9342] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-03-22 13:15:05.925526: E tensorflow/compiler/xla/stream_executor/cuda/cuda_fft.cc:609] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-03-22 13:15:05.925550: E tensorflow/compiler/xla/stream_executor/cuda/cuda_blas.cc:1518] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-03-22 13:15:05.932559: I tensorflow/core/platform/cpu_feature_g

Loading checkpoint shards:   0%|          | 0/6 [00:00<?, ?it/s]

In [18]:
def add_line_numbers(code):
    lines = code.split('\n')
    numbered_lines = [f"{i + 1}. {line}" for i, line in enumerate(lines)]
    return '\n'.join(numbered_lines)
print(add_line_numbers(python_solutions[0:1]["solution_no_comments"].to_list()[0]))

1. def findRelativeRanks(score):
2.     sorted_scores = score[:]
3.     n = len(sorted_scores)
4.     for i in range(n):
5.         for j in range(0, n-i-1):
6.             if sorted_scores[j] < sorted_scores[j+1]:
7.                 sorted_scores[j], sorted_scores[j+1] = sorted_scores[j+1], sorted_scores[j]
8.     rank_dict = {}
9.     for idx, value in enumerate(sorted_scores):
10.         if idx == 0:
11.             rank_dict[value] = "Gold Medal"
12.         elif idx == 1:
13.             rank_dict[value] = "Silver Medal"
14.         elif idx == 2:
15.             rank_dict[value] = "Bronze Medal"
16.         else:
17.             rank_dict[value] = str(idx + 1)
18.     result = []
19.     for s in score:
20.         result.append(rank_dict[s])
21.     return result
22. print(findRelativeRanks([5, 4, 3, 2, 1]))  
23. print(findRelativeRanks([10, 3, 8, 9, 4]))


In [88]:
def get_generated_explanations(programming_language, problem_description, code_solution):
    code_solution_with_lines = add_line_numbers(code_solution.strip())
    explanation_prompt = f"""
        Read the given problem and the proposed {programming_language} solution. For **each line** of the code, provide a detailed explanation. Please ensure the following:

        - **Do not add any extra text** such as test cases, summaries, or comments.
        - Do not merge explanations for multiple lines.
        - Even if two lines seem similar, explain them individually.

        Output format: 
        [Line x]: Explanation: <your explanation>
        
        Problem: {problem_description}
        Code: ```{code_solution_with_lines}```
        """
    explanation_inputs = tokenizer(explanation_prompt, return_tensors="pt", padding=True)
    explanation_outputs = model.generate(input_ids=explanation_inputs['input_ids'], attention_mask=explanation_inputs['attention_mask'], max_length=2048)
    return tokenizer.decode(explanation_outputs[0], skip_special_tokens=True)[len(explanation_prompt)-1:].strip()

In [79]:
columns = ['title', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [61]:
def add_explanations(solutions, programming_language, explanations):
    for index, row in tqdm(solutions.iterrows()):
         explanations.loc[len(explanations)] = [row['title'], get_generated_explanations(programming_language, row['description'], row['solution_no_comments'])]

In [62]:
python_solutions[0:1]

,title,description,difficulty,solution,solution_no_comments
0,Relative Ranks,You are given an integer array `score` of size...,Easy,def findRelativeRanks(score):\n # Step 1: C...,def findRelativeRanks(score):\n sorted_scor...


In [63]:
add_explanations(python_solutions[0:1], Language.PYTHON, python_explanations)

1it [11:53, 713.46s/it]


In [80]:
python_explanations

,title,explanation


In [67]:
print(python_explanations[0:1]["explanation"].to_list()[0])

  [Line 1]: Explanation: Define a function named `findRelativeRanks` that takes a list `score` as its parameter.
[Line 2]: Explanation: Create a copy of the input list `score` and assign it to the variable `sorted_scores`.
[Line 3]: Explanation: Determine the length of the `sorted_scores` list and assign it to the variable `n`.
[Line 4]: Explanation: Start a loop that will iterate `n` times, where `n` is the length of the `sorted_scores` list.
[Line 5]: Explanation: Initialize another loop that will iterate from the start of the list to `n-i-1`. This loop is used to perform the bubble sort algorithm.
[Line 6]: Explanation: Check if the current element `sorted_scores[j]` is less than the next element `sorted_scores[j+1]`.
[Line 7]: Explanation: If the condition in line 6 is true, swap the elements at indices `j` and `j+1` in the `sorted_scores` list.
[Line 8]: Explanation: Create an empty dictionary named `rank_dict` to store the ranks of the athletes.
[Line 9]: Explanation: Start a loo

**Now, I need to run these method for each solution file I generated.**

In [89]:
columns = ['title', 'explanation']
python_explanations = pd.DataFrame([], columns=columns)
c_sharp_explanations = pd.DataFrame([], columns=columns)
typescript_explanations = pd.DataFrame([], columns=columns)

In [90]:
add_explanations(python_solutions, Language.PYTHON, python_explanations)

30it [6:10:57, 741.93s/it] 


In [91]:
add_explanations(c_sharp_solutions, Language.C_SHARP, c_sharp_explanations)

30it [6:59:36, 839.20s/it] 


In [92]:
add_explanations(typescript_solutions, Language.TYPESCRIPT, typescript_explanations)

30it [6:32:44, 785.47s/it]


In [83]:
python_explanations

,title,explanation
0,Relative Ranks,[Line 1]: Explanation: Define a function named...
1,Truncate Sentence,```\n: [Line 1]: Explanation: Define a functio...
2,Perform String Shifts,[Line 1]: Explanation: Define a function named...
3,Invert Binary Tree,``` [Line 1]: Explanation: Defines the TreeNod...
4,Merge Strings Alternately,[Line 1]: Explanation: Define a function named...


# Line Explanation Number Checking
To make sure that each line was explained, I need to check the number of code lines with the number of explanation lines.

In [84]:
def check_explanations_for_lines(code, explanations):
    code_lines = [line for line in code.split('\n') if line.strip()]
    explanation_lines = [line for line in explanations.split('\n') if line.strip().startswith("[Line")]
    
    return len(code_lines) == len(explanation_lines)

In [75]:
check_explanations_for_lines(python_solutions[0:1]["solution_no_comments"].to_list()[0], python_explanations[0:1]["explanation"].to_list()[0])

True

In [85]:
def check_all_explanations(solutions, explanations):
    for index, row in tqdm(solutions.iterrows()):
        if check_explanations_for_lines(row["solution_no_comments"], explanations.loc[index, "explanation"]):
            continue
        else:
            print(row["title"])

In [86]:
check_all_explanations(python_solutions[0:5], python_explanations)

5it [00:00, 2803.68it/s]

Truncate Sentence
Invert Binary Tree


In [93]:
python_explanations.to_csv('datasets/python_14B_explanations.csv', index=False)

In [94]:
c_sharp_explanations.to_csv('datasets/c_sharp_14B_explanations.csv', index=False)

In [95]:
typescript_explanations.to_csv('datasets/typescript_14B_explanations.csv', index=False)